# MiBici Guadalajara — Derived Metrics

Computes age-at-trip and simulates station bike availability over time, using the star schema built in `03_modeling.ipynb`.

## 1. Setup: connect to PostgreSQL

In [3]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

## 2. Calculate age at time of trip

Computed directly in SQL rather than pulling 4.5M rows into pandas and pushing them back. This is a simple join + arithmetic operation, the kind of transformation a database engine handles far more efficiently than round-tripping through Python. `Edad_al_viaje` is added as a new column on `Fact_Trips`.

In [6]:
with engine.connect() as conn:
    conn.execute(text("""
        ALTER TABLE analytics.fact_trips 
        ADD COLUMN IF NOT EXISTS "Edad_al_viaje" INTEGER;
    """))
    conn.execute(text("""
        UPDATE analytics.fact_trips f
        SET "Edad_al_viaje" = (f."Date_Id" / 10000) - u."Año_de_nacimiento"::INTEGER
        FROM analytics.dim_usuario u
        WHERE f."Usuario_Id" = u."Usuario_Id";
    """))
    conn.commit()

check = pd.read_sql("SELECT \"Edad_al_viaje\" FROM analytics.fact_trips LIMIT 5;", con=engine)
print(check)

   Edad_al_viaje
0             43
1             22
2             41
3             37
4             31


## 3. Net flow imbalance (proxy for bike availability issues)

Rather than simulating absolute bike counts, which would require assuming a starting inventory and a rebalancing schedule we have no data to verify, this calculates departures minus arrivals per station, per hour of day. A station that's consistently a strong net "source" during certain hours is one that tends to run low on bikes.

In [9]:
fact_trips = pd.read_sql("SELECT * FROM analytics.fact_trips", con=engine)

departures = fact_trips.groupby(["Origen_Id", "Hora_Inicio"]).size().reset_index(name="departures")
departures = departures.rename(columns={"Origen_Id": "Station_Id", "Hora_Inicio": "Hour"})

fact_trips["Hora_Fin"] = pd.to_datetime(fact_trips["Fin_del_viaje"]).dt.hour
arrivals = fact_trips.groupby(["Destino_Id", "Hora_Fin"]).size().reset_index(name="arrivals")
arrivals = arrivals.rename(columns={"Destino_Id": "Station_Id", "Hora_Fin": "Hour"})

net_flow = departures.merge(arrivals, on=["Station_Id", "Hour"], how="outer").fillna(0)
net_flow["net_flow"] = net_flow["departures"] - net_flow["arrivals"]

print(net_flow.shape)
net_flow.sort_values("net_flow", ascending=False).head(10)

(7789, 5)


,Station_Id,Hour,departures,arrivals,net_flow
948,51,8,12302.0,3690.0,8612.0
3498,194,18,8473.0,1259.0,7214.0
947,51,7,8634.0,2947.0,5687.0
3499,194,19,4787.0,999.0,3788.0
949,51,9,6309.0,2563.0,3746.0
3497,194,17,3938.0,859.0,3079.0
968,52,8,5176.0,2724.0,2452.0
164,11,7,5572.0,3151.0,2421.0
685,38,7,4393.0,2040.0,2353.0
165,11,8,6194.0,4042.0,2152.0


## 3b. Summarize net flow imbalance per station

Reduces the station/hour detail down to one row per station, capturing the peak imbalance hour and its magnitude. This becomes the ranking metric for identifying stations most prone to running low on bikes.

In [13]:
station_peak_imbalance = (
    net_flow.loc[net_flow.groupby("Station_Id")["net_flow"].idxmax()]
    [["Station_Id", "Hour", "net_flow"]]
    .rename(columns={"Hour": "peak_hour", "net_flow": "peak_net_flow"})
    .reset_index(drop=True)
)

print(station_peak_imbalance.shape)
station_peak_imbalance.sort_values("peak_net_flow", ascending=False).head(10)

(371, 3)


,Station_Id,peak_hour,peak_net_flow
47,51,8,8612.0
174,194,18,7214.0
48,52,8,2452.0
8,11,7,2421.0
34,38,7,2353.0
65,69,8,2107.0
82,86,18,2050.0
58,62,8,1527.0
144,164,8,1526.0
162,182,7,1515.0


## 3c. Write net flow tables to analytics

In [16]:
net_flow.to_sql("station_hourly_net_flow", con=engine, schema="analytics", if_exists="replace", index=False)
station_peak_imbalance.to_sql("station_peak_imbalance", con=engine, schema="analytics", if_exists="replace", index=False)

print("station_hourly_net_flow:", net_flow.shape[0], "rows")
print("station_peak_imbalance:", station_peak_imbalance.shape[0], "rows")

station_hourly_net_flow: 7789 rows
station_peak_imbalance: 371 rows


## 4. Station isolation (nearest-neighbor distance)

For each station, finds the closest other station by straight-line distance. This is a purely geographic calculation using all 484 stations' coordinates, independent of trip volume, since isolation is a property of the network's physical layout, not of how the station happens to be used.

Reusing the Haversine distance function from the modeling notebook, since each notebook has its own independent kernel and doesn't share state with others.

In [24]:
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

In [26]:
stations_geo = pd.read_sql("SELECT * FROM analytics.dim_station", con=engine)

lat = stations_geo["latitude"].values
lon = stations_geo["longitude"].values

dist_matrix = haversine_vectorized(lat[:, None], lon[:, None], lat[None, :], lon[None, :])

np.fill_diagonal(dist_matrix, np.inf)

nearest_idx = dist_matrix.argmin(axis=1)
nearest_distance_km = dist_matrix.min(axis=1)

station_isolation = pd.DataFrame({
    "Station_Id": stations_geo["Station_Id"].values,
    "nearest_station_id": stations_geo["Station_Id"].values[nearest_idx],
    "nearest_distance_km": nearest_distance_km
})

print(station_isolation.shape)
station_isolation.sort_values("nearest_distance_km", ascending=False).head(10)

(484, 3)


,Station_Id,nearest_station_id,nearest_distance_km
456,489,430,0.585760
250,261,265,0.558142
155,164,180,0.446413
131,140,221,0.445272
146,155,161,0.440646
222,231,361,0.432492
458,491,443,0.425311
451,483,484,0.411625
216,225,224,0.406754
278,289,96,0.401455


## 4b. Check the full distribution, not just the extremes

The top-10 "most isolated" stations all sit under 0.6 km from their nearest neighbor, suggesting the network may be too geographically dense for nearest-neighbor distance alone to meaningfully differentiate stations. Checking the full distribution before deciding whether this metric needs adjusting.

In [29]:
print(station_isolation["nearest_distance_km"].describe())

count    484.000000
mean       0.230899
std        0.077178
min        0.000000
25%        0.189913
50%        0.232943
75%        0.277907
max        0.585760
Name: nearest_distance_km, dtype: float64


## 4c. Check for near-duplicate station coordinates

The minimum nearest-neighbor distance is exactly 0, worth confirming whether this is a genuine data issue (duplicate or mislabeled coordinates) before using this metric further.

In [32]:
near_duplicates = station_isolation[station_isolation["nearest_distance_km"] < 0.01]
print(near_duplicates)

     Station_Id  nearest_station_id  nearest_distance_km
111         116                 248             0.000000
129         138                 249             0.000000
238         248                 116             0.000000
239         249                 138             0.000000
294         310                 390             0.004855
373         390                 310             0.004855


## 4c-2. Investigate the duplicate-coordinate station pairs

Two station pairs share identical coordinates. Checking their names and capacities to distinguish between a genuine data error (accidental duplicate entry) and a real-world case of two physically co-located docks (e.g., an expansion added under a new ID at the same location).

In [58]:
duplicate_ids = [111, 116, 129, 138, 238, 248, 239, 249]
# Using the actual Station_Id values, not the DataFrame's row index
duplicate_station_ids = [116, 248, 138, 249]

print(stations_geo[stations_geo["Station_Id"].isin(duplicate_station_ids)]
      [["Station_Id", "name", "latitude", "longitude", "dpcapacity"]])

     Station_Id                                           name  latitude  \
111         116  Ex-ZPN-026 C. 16 Sep. / C. Fray A. de Segovia  20.68692   
129         138                                     Ex-ZPN-044  20.69705   
238         248  (GDL-180) C. Montes Pirineos/Salvador Quevedo  20.68692   
239         249  (GDL-181) C.Ignacio Rmrz / Av Plan de San Lui  20.69705   

     longitude  dpcapacity  
111 -103.33467           0  
129 -103.36322           0  
238 -103.33467          19  
239 -103.36322          15  


## 4c-3. Check whether decommissioned stations have real trip history

Stations with dpcapacity = 0 represent decommissioned locations. Checking whether they still appear in Fact_Trips matters: if they do, those trips likely occurred while the station was still active, meaning the current (zero) capacity shouldn't be applied retroactively to that historical activity. It also affects the density metric, a decommissioned station with 0 real capacity shouldn't count as a "nearby station" a rider could actually use today.

In [61]:
zero_capacity_ids = stations_geo[stations_geo["dpcapacity"] == 0]["Station_Id"].tolist()
print("Stations with 0 capacity:", zero_capacity_ids)

trips_at_zero_cap = fact_trips[
    fact_trips["Origen_Id"].isin(zero_capacity_ids) | fact_trips["Destino_Id"].isin(zero_capacity_ids)
]
print("Trips referencing a 0-capacity station:", len(trips_at_zero_cap))

Stations with 0 capacity: [99, 109, 116, 138]
Trips referencing a 0-capacity station: 0


## 4c-4. Documented limitation: station file has no snapshot date

Stations 99, 109, 116, and 138 show dpcapacity = 0 in the station metadata file, and zero trips reference them anywhere in the 2025 trip data, consistent with them being inactive. However, a live check of MiBici's station map (September 2026) shows all four as currently active.

This reveals a limitation of the source data: the station file is a single static snapshot with no timestamp indicating what date it reflects. It's therefore unclear whether these stations were inactive for all of 2025, part of 2025, or whether the mismatch is unrelated to the 2025 analysis window entirely. No data exists to reconstruct the actual timeline, so this is documented as a known constraint rather than resolved with an assumption.

For this analysis, these 4 stations are excluded from network density calculations, since the density metric is meant to reflect coverage relevant to the 2025 trip patterns being analyzed, which show no activity at these locations. Their current (2026) active status is noted here for transparency but does not change how 2025 data is treated.

In [66]:
dim_station_updated = stations_geo.copy()
dim_station_updated["had_2025_activity"] = ~dim_station_updated["Station_Id"].isin(zero_capacity_ids)

dim_station_updated.to_sql("dim_station", con=engine, schema="analytics", if_exists="replace", index=False)

print(dim_station_updated["had_2025_activity"].value_counts())

had_2025_activity
True     480
False      4
Name: count, dtype: int64


## 4c-5. Confirm the 0-capacity stations aren't inflating other stations' density counts

Even if these 4 stations don't appear at the low end of the density ranking, they were still included in the distance matrix and could be counted as a "nearby station" for other real, active stations, artificially inflating those stations' density scores. Checking directly before concluding this doesn't need a fix.

In [70]:
print("Density value for the 4 no-2025-activity stations themselves:")
print(station_density[station_density["Station_Id"].isin(zero_capacity_ids)])
print()

zero_cap_positions = stations_geo[stations_geo["Station_Id"].isin(zero_capacity_ids)].index.tolist()
affected = (dist_matrix[:, zero_cap_positions] <= radius_km).any(axis=1)

print("Number of OTHER stations that count one of these 4 as a nearby neighbor:", affected.sum())

Density value for the 4 no-2025-activity stations themselves:
     Station_Id  stations_within_400m
94           99                     7
104         109                     5
111         116                     7
129         138                     4

Number of OTHER stations that count one of these 4 as a nearby neighbor: 19


## 4c-note. Why nearest-neighbor distance was abandoned

Nearest-neighbor distance was explored first, but the results (mean 231m, max 586m across all 484 stations) showed almost no variation, the deployed network is geographically dense and fairly uniform within its coverage area, so this metric couldn't meaningfully distinguish well-served stations from poorly-served ones. A visual check against MiBici's live station map confirmed why: stations cluster tightly along a defined corridor, with no coverage at all outside it, rather than a gradual gradient of isolation. Station density (count of neighbors within a realistic walking radius, section 4d) captures this pattern far better and is used as the actual metric going forward.

## 4d. Station density within walking radius (400m)

Since nearest-neighbor distance shows little variation across the network (stations are uniformly close together within the deployed corridor), this instead counts how many other stations fall within a 400m radius, roughly a 5-minute walk per the EAM 2023 survey. This better reflects the pattern visible on MiBici's own station map: dense clustering within a defined corridor, with no stations at all outside it, rather than a gradient of isolation.

In [56]:
within_radius = (dist_matrix <= radius_km).sum(axis=1)  # diagonal already set to inf earlier, so self is naturally excluded

station_density = pd.DataFrame({
    "Station_Id": stations_geo["Station_Id"].values,
    "stations_within_400m": within_radius
})

print(station_density["stations_within_400m"].describe())
station_density.sort_values("stations_within_400m").head(10)

count    484.000000
mean       3.909091
std        2.165034
min        0.000000
25%        2.000000
50%        4.000000
75%        5.000000
max       12.000000
Name: stations_within_400m, dtype: float64


,Station_Id,stations_within_400m
131,140,0
458,491,0
278,289,0
250,261,0
146,155,0
155,164,0
451,483,0
222,231,0
216,225,0
456,489,0


## 4d (corrected). Station density, excluding stations with no 2025 activity

The 4 stations with no recorded 2025 activity are excluded as candidate neighbors, since they weren't part of the network riders could actually use during the analysis period. This affects 19 other stations whose density counts were previously inflated by counting one of these 4 as a neighbor.

In [73]:
active_mask = ~stations_geo["Station_Id"].isin(zero_capacity_ids).values

dist_matrix_active = dist_matrix.copy()
dist_matrix_active[:, ~active_mask] = np.inf  # exclude no-2025-activity stations as counted neighbors

within_radius_corrected = (dist_matrix_active <= radius_km).sum(axis=1)

station_density = pd.DataFrame({
    "Station_Id": stations_geo["Station_Id"].values,
    "stations_within_400m": within_radius_corrected,
    "had_2025_activity": active_mask
})

print(station_density["stations_within_400m"].describe())
print()
print("Confirm the 19 previously-affected stations changed:")
print(station_density[station_density["Station_Id"].isin(
    stations_geo.iloc[np.where(affected)[0]]["Station_Id"]
)].head(19))

count    484.000000
mean       3.861570
std        2.128251
min        0.000000
25%        2.000000
50%        4.000000
75%        5.000000
max       12.000000
Name: stations_within_400m, dtype: float64

Confirm the 19 previously-affected stations changed:
     Station_Id  stations_within_400m  had_2025_activity
94           99                     6              False
102         107                     4               True
105         110                     4               True
107         112                     4               True
108         113                     5               True
109         114                     3               True
111         116                     6              False
169         178                     7               True
172         181                     6               True
236         246                     5               True
237         247                     5               True
238         248                     5               True
23

## 5. Write derived metrics to analytics

In [78]:
station_density.to_sql("station_density", con=engine, schema="analytics", if_exists="replace", index=False)

print("station_density:", station_density.shape[0], "rows")

station_density: 484 rows


## 6. Phase 5 summary

Derived metrics added to the analytics schema:
- `Edad_al_viaje` added directly to Fact_Trips (age at time of trip, computed per-trip rather than as a fixed value)
- `station_hourly_net_flow` and `station_peak_imbalance`: directional proxy for bike availability issues, replacing an original plan to simulate absolute occupancy (which would have required unverifiable baseline/reset assumptions)
- `station_density`: count of other active stations within a 400m/~5min walking radius (per EAM 2023), excluding 4 stations with no recorded 2025 activity despite currently being active as of September 2026, a documented limitation of the station file having no snapshot date

Known limitations documented throughout: birth-year-only age approximation (no exact birthdate), no truck rebalancing data (hence the pivot away from absolute occupancy), and the station file's undated snapshot creating ambiguity around the 4 currently-active-but-2025-inactive stations.

## Methodology notes

**Walking radius assumption (400m):** Station density is calculated using a 400m 
radius, approximating a 5-minute walk. This threshold is informed by Jalisco's 
Encuesta Anual de Movilidad (EAM) 2023, which found that when a station has no 
bikes available, most users report walking to a nearby station rather than 
switching to another mode, with roughly 5 minutes being a commonly reported 
tolerance before choosing a different option (public transit, rideshare, etc.).